In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import pymc3 as pm
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        break
#print(os.path.join(dirname, filename))

# You can write up to 5GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session


/usr/local/lib/python3.11/dist-packages/theano/scalar/basic.py:2323: FutureWarning: In the future `np.bool` will be defined as the corresponding NumPy scalar.
  self.ctor = getattr(np, o_type.dtype)


AttributeError: module 'numpy' has no attribute 'bool'.
`np.bool` was a deprecated alias for the builtin `bool`. To avoid this error in existing code, use `bool` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.bool_` here.
The aliases was originally deprecated in NumPy 1.20; for more details and guidance see the original release note at:
    https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations

In [2]:
# Preprocessing
# train is resused/editted. train_raw is left alone
train = pd.read_csv('../input/osic-pulmonary-fibrosis-progression/train.csv')
train_raw = pd.read_csv('../input/osic-pulmonary-fibrosis-progression/train.csv')
test = pd.read_csv('../input/osic-pulmonary-fibrosis-progression/test.csv')

train.drop(train[train.Patient == 'ID00197637202246865691526'].index, inplace=True)

# exclude_test_patient_data_from_trainset = False

# if exclude_test_patient_data_from_trainset:
#     train = train[~train['Patient'].isin(test['Patient'].unique())]

train = pd.concat([train, test], axis=0, ignore_index=True)\
    .drop_duplicates()
le_id = LabelEncoder()
train['PatientID'] = le_id.fit_transform(train['Patient'])

train.head()


NameError: name 'LabelEncoder' is not defined

In [3]:
def add_baselines(data):
    aux = data[['Patient', 'Weeks']].groupby('Patient')\
        .min().reset_index()
    aux = pd.merge(aux, data[['Patient', 'Weeks', 'FVC']], how='left', 
                   on=['Patient', 'Weeks'])
    aux = aux.groupby('Patient').mean().reset_index()
    aux['Weeks'] = aux['Weeks'].astype(int)
    aux['FVC'] = aux['FVC'].astype(int)
    data = pd.merge(data, aux, how='left', on='Patient', suffixes=('', '_base'))
    return data

train = add_baselines(train)
test = add_baselines(test)
train.head()


,Patient,Weeks,FVC,Percent,Age,Sex,SmokingStatus,Weeks_base,FVC_base
0,ID00133637202223847701934,-2,3195,92.856312,83,Male,Never smoked,-2,3195
1,ID00133637202223847701934,2,3203,93.088817,83,Male,Never smoked,-2,3195
2,ID00133637202223847701934,4,3097,90.008138,83,Male,Never smoked,-2,3195
3,ID00133637202223847701934,6,3171,92.158800,83,Male,Never smoked,-2,3195
4,ID00133637202223847701934,8,3115,90.531272,83,Male,Never smoked,-2,3195


In [4]:
def patient_class(row):
    if row['Sex'] == 'Male':
        if row['SmokingStatus'] == 'Currently smokes':
            return 0
        elif row['SmokingStatus'] == 'Ex-smoker':
            return 1
        elif row['SmokingStatus'] == 'Never smoked':
            return 2
    else:
        if row['SmokingStatus'] == 'Currently smokes':
            return 3
        elif row['SmokingStatus'] == 'Ex-smoker':
            return 4
        elif row['SmokingStatus'] == 'Never smoked':
            return 5

train['Class'] = train.apply(patient_class, axis=1)
test['Class'] = test.apply(patient_class, axis=1)
test.head()
train.loc[train['Patient']=='ID00007637202177411956430']['Class'].max()


1

In [5]:
PatientID = train['Patient'].values
fvc_b = train.groupby('Patient').first()['FVC_base']
fvc_b.values


array([2315, 3660, 3523, 3326, 3418, 3807, 3042, 2100, 2297, 1536, 2903,
       2884, 2472, 2506, 5045, 2728, 3946, 3791, 3313, 1426, 1697, 2489,
       3255, 3969, 2301, 3456, 2810, 2359, 2547, 1641, 2298, 3170, 1860,
       2918, 3367, 2508, 3562, 3695, 4916, 2934, 2553, 2377, 2479, 2914,
       2073, 1995, 2389, 2548, 2933, 2917, 2581, 1674, 3559, 2466, 2375,
       1677, 2220, 2253, 1690, 3327, 2582, 3195, 2629, 3808, 3239, 4510,
       3793, 4124, 3247, 2841, 2102, 4562, 3186, 2520, 1970, 2603, 2648,
       3009, 2541, 3130, 3096, 4291, 1728, 3849, 2677, 4311, 2379, 2846,
       3035, 2869, 2658, 2439, 6399, 1401, 2644, 2992, 1583, 1885, 3518,
       1909, 3829, 2080, 2061, 3061, 1399, 1853, 1272, 3255, 2075, 1563,
       1938, 1015, 3020, 2080, 3107, 4141, 1965, 1411, 2276, 2283, 4051,
       2327, 2223, 1880, 2109, 2984, 3170, 3240, 1879, 2375, 4143, 2460,
       1412, 2805, 3135, 3043, 2569, 3959, 3574, 2345, 2849, 2390, 1637,
       2066, 2808, 4418, 2030, 3107, 3191, 2421, 13

In [6]:
def model_fit(data, examine=True):
    n_patients = data['Patient'].nunique()
    FVC_obs = data['FVC'].values
    Weeks = data['Weeks'].values
    #X = train[['Weeks', 'Male', 'ExSmoker', 'CurrentlySmokes', 'Percent_base']].values
    PatientID = data['PatientID'].values
    patient_class = data['Class'].values
    FVC_b = data.groupby('PatientID').first()['FVC_base']
    w_b = data.groupby('PatientID').first()['Weeks_base']

    with pm.Model() as model:
        # create shared variables that can be changed later on
        FVC_obs_shared = pm.Data("FVC_obs_shared", FVC_obs)
        Weeks_shared = pm.Data('Weeks_shared', Weeks)
        PatientID_shared = pm.Data('PatientID_shared', PatientID)
        patient_class_shared = pm.Data('patient_class_shared', patient_class)
        FVC_b_shared = pm.Data('FVC_b_shared', FVC_b)
        w_b_shared= pm.Data('w_b_shared', w_b)

        mu_a_base = pm.Normal('mu_a_base', mu=0., sigma=100)
        mu_a = FVC_b_shared + mu_a_base * w_b_shared
        
        #mu_a = pm.Normal('mu_a', mu=1700., sigma=400)
        sigma_a = pm.HalfNormal('sigma_a', 1000.)
        mu_b = pm.Normal('mu_b', mu=-4., sigma=1)
        sigma_b = pm.HalfNormal('sigma_b', 5.)
        #mu_c = pm.Normal('mu_c', mu=0, sigma=100)
        #sigma_c = pm.HalfNormal('sigma_c', 100)

        a = pm.Normal('a', mu=mu_a, sigma=sigma_a, shape=n_patients)
        b = pm.Normal('b', mu=mu_b, sigma=sigma_b, shape=n_patients)
        #c = pm.Normal('c', mu=mu_c, sigma=sigma_c, shape=n_patients)

        # Model error
        #sigma = pm.HalfNormal('sigma', 150.)
        sigma = pm.HalfNormal('sigma', 150., shape=6)
        
        FVC_est = a[PatientID_shared] + b[PatientID_shared] * Weeks_shared

        # Data likelihood
        FVC_like = pm.Normal('FVC_like', mu=FVC_est,
                             sigma=sigma[patient_class_shared], observed=FVC_obs_shared)

        # Fitting the model
        trace = pm.sample(2000, tune=2000, target_accept=.9, init="adapt_diag")
        
    if examine:
        with model:
            pm.traceplot(trace);
            
    return model, trace


In [7]:
# with model_a:
#     pm.traceplot(trace_a);


In [8]:
def generate_template(data):
    pred_template = []
    for i, patient in enumerate(data['Patient'].unique()):
        df = pd.DataFrame(columns=['PatientID', 'Weeks', 'Patient', 'Class'])
        df['Weeks'] = np.arange(-12, 134)
        df['Patient'] = patient
        df['Class'] = data.loc[data['Patient']==patient]['Class'].max()
        pred_template.append(df)
    pred_template = pd.concat(pred_template, ignore_index=True)
    pred_template['PatientID'] = le_id.transform(pred_template['Patient'])
    return pred_template


In [9]:
template_train_test = generate_template(test)
template_train_test.head()


NameError: name 'le_id' is not defined

In [10]:
def model_predict(model, trace, template):
    with model:
        pm.set_data({
            "PatientID_shared": template['PatientID'].values.astype(int),
            "Weeks_shared": template['Weeks'].values.astype(int),
            "FVC_obs_shared": np.zeros(len(template)).astype(int),
            "patient_class_shared": template['Class'].values.astype(int),
        })
        post_pred = pm.sample_posterior_predictive(trace)
    df = pd.DataFrame(columns=['Patient', 'Weeks', 'FVC_pred', 'sigma'])
    df['Patient'] = le_id.inverse_transform(template['PatientID'])
    df['Weeks'] = template['Weeks']
    df['FVC_pred'] = post_pred['FVC_like'].T.mean(axis=1)
    df['sigma'] = post_pred['FVC_like'].T.std(axis=1)
    df['FVC_inf'] = df['FVC_pred'] - df['sigma']
    df['FVC_sup'] = df['FVC_pred'] + df['sigma']
    df = pd.merge(df, train[['Patient', 'Weeks', 'FVC']], how='left', on=['Patient', 'Weeks'])
    #if exclude_test_patient_data_from_trainset:
    #df=df.append(train_raw[train_raw['Patient'].isin(test['Patient'].unique())][['Patient', 'Weeks', 'FVC']])
    df = df.rename(columns={'FVC': 'FVC_true'})
    return df


In [11]:
def examine_predictions(data):
    n = (data['Patient'].nunique())+1
    f, axes = plt.subplots((n//3)+1, 3, figsize=(15, 5*((n//3)+1)))
    #for i, patient in enumerate(np.random.choice(data['Patient'].unique(), size=3, replace=False)):
    for i, patient in enumerate(data['Patient'].unique()):
        ax = axes[i//3, i%3]
        df = data[data['Patient'] == patient]
        x = df['Weeks']
        ax.set_title(patient)
        ax.plot(x, df['FVC_true'], 'o')
        ax.plot(x, df['FVC_pred'])
        ax = sns.regplot(x, df['FVC_true'], ax=ax, ci=None, line_kws={'color':'red'})
        ax.fill_between(x, df["FVC_inf"], df["FVC_sup"],alpha=0.5, color='#ffcd3c')
        ax.set_ylabel('FVC')
    axes[n//3,n%3].plot()


In [12]:
def evaluate_predictions(df, use_only_last_3_measures=False, examine=False):
    if use_only_last_3_measures:
        y = df.dropna().groupby('Patient').tail(3)
    else:
        y = df.dropna()

    rmse = ((y['FVC_pred'] - y['FVC_true']) ** 2).mean() ** (1/2)
    mae = (y['FVC_pred'] - y['FVC_true']).abs()
    mae_mean = (np.sqrt((y['FVC_pred'] - y['FVC_true']) ** 2)).mean()
    mae_sd = (np.sqrt((y['FVC_pred'] - y['FVC_true']) ** 2)).mean()
    mae_max = (np.sqrt((y['FVC_pred'] - y['FVC_true']) ** 2)).mean()
    #print(f'RMSE (mean): {rmse:.1f} ml')
    #print(f'MAE (mean): {rmse:.1f} ml')
    sigma_c = y['sigma'].values*1.5
    sigma_c[sigma_c < 70] = 70
    delta = (y['FVC_pred'] - y['FVC_true']).abs()
    delta[delta > 1000] = 1000
    lll = - np.sqrt(2) * delta / sigma_c - np.log(np.sqrt(2) * sigma_c)

    #print(f'Laplace Log Likelihood: {lll.mean():.4f}')
    y['sigma_c'] = y['sigma']
    y['sigma_c'].values[y['sigma_c'].values < 70] = 70
    y['delta_c'] = (y['FVC_pred'] - y['FVC_true']).abs()
    y['delta_c'].values[y['delta_c'].values > 1000] = 1000
    y['main_loss'] = y['delta_c']/y['sigma_c']
    if examine:
        plt.hist(y['main_loss'], bins=100)

    #bad_patients = y[y['main_loss'] > 2.5]
    #bad_patients
    #bad_patients['Patient'].unique()

    #plt.scatter(delta, sigma_c);
    #plt.xlabel("delta");
    #plt.ylabel("sigma_c");
    return lll.mean()


In [13]:
def evaluation_cycle(train, valid, examine_trace=True, examine_preds=True):
    # Fit model
    print("Fit model ...")
    model,trace = model_fit(train, examine=examine_trace)
    print("")

    # Model predictions with training data
    print("Examine true vs predictions for training data ...")
    template_train = generate_template(train)
    template_train.head()
    pred_train = model_predict(model,trace,template_train)
    if examine_preds:
        examine_predictions(pred_train)
    lll_train = evaluate_predictions(pred_train)

    # Model predictions with validation data (unseen)
    if valid is not None:
        print("Examine true vs predictions for validation data ...")
        template_valid = generate_template(valid)
        pred_valid = model_predict(model,trace,template_valid)
        if examine_preds:
            examine_predictions(pred_valid)
        lll_valid = evaluate_predictions(pred_valid)
    return pred_train, pred_valid, lll_train, lll_valid


In [14]:
#Evaluate model
for fold in range(0):
    examine_lll = True

    all_patients = train['Patient'].unique()
    #validation_patients = ['ID00007637202177411956430', 'ID00009637202177434476278', 'ID00010637202177584971671']
    validation_patients = np.random.choice(all_patients, size=20, replace=False)
    df_valid = train[train['Patient'].isin(validation_patients)]
    df_train = train[~train['Patient'].isin(validation_patients)]

    df_valid_first_readings = df_valid.groupby('Patient').head(1)
    df_train = pd.concat([df_train, df_valid_first_readings], axis=0, ignore_index=True)

    print(f'Fold: {fold}')
    pred_train, pred_valid, lll_train, lll_valid = evaluation_cycle(df_train, df_valid, examine_trace=True, examine_preds=True)

    print(f'Laplace Log Likelihoods for fold: {fold}')
    print(f'Training:     {lll_train:.4f}')
    print(f'Validation:   {lll_valid:.4f}')
    print("")

    evaluate_predictions(pred_train, use_only_last_3_measures=True, examine=examine_lll)
    evaluate_predictions(pred_valid, use_only_last_3_measures=True, examine=examine_lll)


In [15]:
# Fit model using all of training data
print("Fit model ...")
model,trace = model_fit(train, examine=False)
print("")

# Make predictions for test data
print("Make predictions for test data ...")
template_test = generate_template(test)
template_test.head()
pred_test = model_predict(model,trace,template_test)
#examine_predictions(pred_test)

# Prepare final submission
final = pd.DataFrame(columns=['Patient_Week', 'FVC', 'Confidence'])
final['Patient_Week'] = pred_test['Patient'] + '_' + pred_test['Weeks'].astype(str)
final['FVC'] = pred_test['FVC_pred']
final['Confidence'] = pred_test['sigma']
final.head()
final.to_csv('submission.csv', index=False)
print(final.shape)
final.head()


Fit model ...


KeyError: 'PatientID'

In [16]:
# pred_template = []
# for p in test['Patient'].unique():
#     df = pd.DataFrame(columns=['PatientID', 'Weeks'])
#     df['Weeks'] = np.arange(-12, 134)
#     df['Patient'] = p
#     pred_template.append(df)
# pred_template = pd.concat(pred_template, ignore_index=True)
# pred_template['PatientID'] = le_id.transform(pred_template['Patient'])

# with model_a:
#     pm.set_data({
#         "PatientID_shared": pred_template['PatientID'].values.astype(int),
#         "Weeks_shared": pred_template['Weeks'].values.astype(int),
#         "FVC_obs_shared": np.zeros(len(pred_template)).astype(int),
#     })
#     post_pred = pm.sample_posterior_predictive(trace_a)


In [17]:
# df = pd.DataFrame(columns=['Patient', 'Weeks', 'Patient_Week', 'FVC', 'Confidence'])
# df['Patient'] = pred_template['Patient']
# df['Weeks'] = pred_template['Weeks']
# df['Patient_Week'] = df['Patient'] + '_' + df['Weeks'].astype(str)
# df['FVC'] = post_pred['FVC_like'].T.mean(axis=1)
# df['Confidence'] = post_pred['FVC_like'].T.std(axis=1)
# final = df[['Patient_Week', 'FVC', 'Confidence']]
# final.to_csv('submission.csv', index=False)
# print(final.shape)
# final.head()
